In [8]:
from pathlib import Path
import sys

import mlflow
import mlflow.pytorch
import numpy as np
import torch

In [9]:
project_root = Path(
    "/home/techstor1/a/sbrunswi/projects/ISO-pHNN"
)

assert (project_root / "model.py").is_file()
sys.path.insert(0, str(project_root))

import model

In [10]:
data_dir = Path(
    "/home/techstor1/a/sbrunswi/projects/ISO-pHNN/"
    "experiments-PH-SINDY/results/20261112_223405/data"
)

database = data_dir.parent / "mlflow.db"
dataset = data_dir / "spring_32train_100val_seed1000.npy"

# Check paths before connecting.
assert database.is_file(), database
assert dataset.is_file(), dataset

mlflow.set_tracking_uri(f"sqlite:///{database}")

run_id = "5cf158bf17d8440cb6d6d7957cb89f4d"

model = mlflow.pytorch.load_model(
    f"runs:/{run_id}/model",
    map_location="cpu",
)

data = np.load(dataset, allow_pickle=True).item()




In [11]:
for n in [4, 8, 16, 32, 64]:
    path = data_dir / f"spring_{n}train_100val_seed1000.npy"
    data = np.load(path, allow_pickle=True).item()

    print(f"\n{n} training trajectories")

    for key in ["X", "u", "xdot", "y",
                "X_val", "u_val", "xdot_val", "y_val"]:
        array = data[key]
        print(
            f"  {key:10s} shape={array.shape}, "
            f"finite={np.isfinite(array).all()}"
        )

    if reference is None:
        reference = data
    else:
        for key in ["X_val", "u_val", "xdot_val", "y_val"]:
            print(
                f"  {key} identical to first dataset:",
                np.array_equal(data[key], reference[key]),
            )


4 training trajectories
  X          shape=(3996, 4), finite=True
  u          shape=(3996, 2), finite=True
  xdot       shape=(3996, 4), finite=True
  y          shape=(3996, 2), finite=True
  X_val      shape=(999900, 4), finite=True
  u_val      shape=(999900, 2), finite=True
  xdot_val   shape=(999900, 4), finite=True
  y_val      shape=(999900, 2), finite=True

8 training trajectories
  X          shape=(7992, 4), finite=True
  u          shape=(7992, 2), finite=True
  xdot       shape=(7992, 4), finite=True
  y          shape=(7992, 2), finite=True
  X_val      shape=(999900, 4), finite=True
  u_val      shape=(999900, 2), finite=True
  xdot_val   shape=(999900, 4), finite=True
  y_val      shape=(999900, 2), finite=True
  X_val identical to first dataset: True
  u_val identical to first dataset: True
  xdot_val identical to first dataset: True
  y_val identical to first dataset: True

16 training trajectories
  X          shape=(15984, 4), finite=True
  u          shape=(15984,

In [12]:
# Function to compute training and validation nme
def derivative_nmae(model, states, inputs, targets, batch_size=512):
    predictions = []

    for start in range(0, len(states), batch_size):
        stop = start + batch_size

        x = torch.as_tensor(states[start:stop], dtype=torch.float32)
        u = torch.as_tensor(inputs[start:stop], dtype=torch.float32)

        # Hamiltonian gradients require autograd during prediction.
        with torch.enable_grad():
            xdot_pred, _ = model(x, u)

        predictions.append(xdot_pred.detach().cpu())

    predictions = torch.cat(predictions, dim=0)
    targets = torch.as_tensor(targets, dtype=torch.float32)

    # Compute normalization across the complete split.
    std = targets.std(dim=0)
    per_component = (
        (predictions - targets).abs() / (std + 1e-6)
    ).mean(dim=0)

    return per_component.mean().item(), per_component.numpy()

In [13]:
train_nmae, train_components = derivative_nmae(
    model, data["X"], data["u"], data["xdot"]
)

val_nmae, val_components = derivative_nmae(
    model, data["X_val"], data["u_val"], data["xdot_val"]
)

logged_nmae = mlflow.get_run(run_id).data.metrics["mae_rel"]

print("Training NMAE:          ", train_nmae)
print("Validation NMAE:        ", val_nmae)
print("Logged validation NMAE: ", logged_nmae)

print("Training per component:  ", train_components)
print("Validation per component:", val_components)

print(
    "Validation matches:",
    np.isclose(val_nmae, logged_nmae, rtol=1e-4, atol=1e-6),
)

Training NMAE:           0.001080942340195179
Validation NMAE:         0.07562524080276489
Logged validation NMAE:  0.07562524825334549
Training per component:   [0.00052869 0.00167127 0.00052263 0.00160118]
Validation per component: [0.02947432 0.10183324 0.01399632 0.1571971 ]
Validation matches: True
